# MPI — Formas de Send/Receive, Curingas e Medição de Tempo

**Disciplina:** DCE540 — Computação Paralela e Distribuída
**Curso:** Bacharelado em Ciência da Computação — UNIFAL-MG

## Objetivos da aula

- Partir do par bloqueante básico `MPI_Send` / `MPI_Recv` e entender exatamente o que "bloqueante" significa.
- Conhecer os **quatro modos de envio bloqueante** do MPI — `MPI_Send` (padrão), `MPI_Ssend` (síncrono), `MPI_Bsend` (bufferizado) e `MPI_Rsend` (pronto) — e as diferenças semânticas entre eles.
- Aprender comunicação **não bloqueante** com `MPI_Isend` / `MPI_Irecv`, `MPI_Request` e `MPI_Wait`.
- Usar os curingas `MPI_ANY_SOURCE` e `MPI_ANY_TAG` para receber mensagens sem saber a priori quem vai enviar nem com que tag.
- Medir tempo de execução com `MPI_Wtime` e `MPI_Wtick`, e entender por que `MPI_Barrier` é necessário antes de iniciar a medição.

> Este notebook foi preparado para o Google Colab e trabalha em **C**, a API "nativa" do MPI — a mesma linguagem do código que deu origem a esta aula. Cada exemplo é escrito em um arquivo (`%%writefile`), compilado com `mpicc` e executado com `mpirun`, simulando múltiplos processos em um único ambiente.

> **Observação:** o Colab roda em uma única máquina (um único nó). Os exemplos abaixo simulam múltiplos processos MPI *na mesma máquina*, suficiente para aprender a sintaxe e a semântica da comunicação ponto a ponto, mas sem representar a rede real de um cluster distribuído.

## 0. Preparando o ambiente

Instalamos o **Open MPI** (compilador `mpicc` e executor `mpirun`).

> **Nota sobre o `--oversubscribe`:** as máquinas virtuais do Google Colab costumam ter poucos núcleos de CPU (muitas vezes apenas 2). Como vários exemplos abaixo usam `-np 4` ou mais processos, o Open MPI recusaria a execução por padrão, informando que não há "slots" suficientes. A flag `--oversubscribe`, presente em todos os comandos `mpirun` deste notebook, instrui o Open MPI a permitir mais processos do que núcleos físicos (eles serão intercalados pelo sistema operacional). Em um cluster real, com núcleos suficientes para o `-np` desejado, essa flag não é necessária.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq libopenmpi-dev openmpi-bin
print("Instalação concluída.")

## 1. Comunicação ponto a ponto bloqueante — `MPI_Send` / `MPI_Recv`

O programa abaixo é o exemplo clássico de "anel de saudações" (baseado em Pacheco, *An Introduction to Parallel Programming*): cada processo monta uma mensagem de texto, envia para o **vizinho seguinte** (`(rank + 1) % size`) e recebe do **vizinho anterior** (`(rank - 1 + size) % size`), formando um anel fechado.

**Assinaturas (C):**
```c
int MPI_Send(const void* buf, int count, MPI_Datatype datatype,
             int dest, int tag, MPI_Comm comm);

int MPI_Recv(void* buf, int count, MPI_Datatype datatype,
             int source, int tag, MPI_Comm comm, MPI_Status* status);
```
- `buf` / `count` / `datatype`: endereço, quantidade e tipo dos elementos da mensagem.
- `dest` (`Send`) / `source` (`Recv`): rank do processo destino/origem.
- `tag`: rótulo inteiro definido pelo programador, usado para diferenciar mensagens entre os mesmos dois processos.
- `comm`: comunicador (em geral `MPI_COMM_WORLD`).
- `status` (`Recv`): estrutura preenchida com informações sobre a mensagem recebida (quem enviou, com que tag, etc. — ver Seção 4). Quando essa informação não interessa, usa-se `MPI_STATUS_IGNORE`, como no exemplo abaixo.

`MPI_Recv` é sempre bloqueante: só retorna quando a mensagem completa já está no buffer. `MPI_Send`, o modo **padrão**, é bloqueante no sentido de que o buffer `greeting` pode ser reaproveitado com segurança assim que a chamada retorna — mas a norma do MPI **não define** se isso significa "esperou o `Recv` correspondente" ou "copiou para um buffer interno e retornou na hora". Essa decisão fica a critério de cada implementação (ver o quadro de atenção logo após o exemplo).

In [ ]:
%%writefile greetings.c
#include <stdio.h>
#include <string.h>  /* For strlen             */
#include <mpi.h>     /* For MPI functions, etc */

const int MAX_STRING = 100;

int main(void) {
   char     greeting[MAX_STRING];  /* String storing message */
   int        size;               /* Number of processes    */
   int        rank;               /* My process rank        */

   /* Start up MPI */
   MPI_Init(NULL, NULL);
   /* Get the number of processes */
   MPI_Comm_size(MPI_COMM_WORLD, &size);
   /* Get my rank among all the processes */
   MPI_Comm_rank(MPI_COMM_WORLD, &rank);

   /* Create message */
   sprintf(greeting, "Greetings from process %d of %d!",  rank, size);
   /* Send message to process 0 */
   MPI_Send(greeting, strlen(greeting) + 1, MPI_CHAR, (rank + 1) % size, 0, MPI_COMM_WORLD);
   /* Receive comm_smessage from process q */
   MPI_Recv(greeting, MAX_STRING, MPI_CHAR, (rank - 1 + size) % size, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
   /* Print message from process q */
   printf("%s\n", greeting);

   /* Shut down MPI */
   MPI_Finalize();

   return 0;
}

In [ ]:
!mpicc -o greetings greetings.c
!mpirun --allow-run-as-root --oversubscribe -np 4 ./greetings

> **Por que este código não trava?** Todo processo executa `MPI_Send` **antes** do `MPI_Recv`, o que em tese poderia causar um impasse (*deadlock*): se `MPI_Send` só retornasse depois que o destino chamasse `MPI_Recv`, ninguém chamaria `Recv` primeiro e todos ficariam parados esperando uns aos outros. Na prática o programa funciona porque, para mensagens **pequenas**, a maioria das implementações de MPI usa um protocolo *eager*: copia a mensagem para um buffer interno e devolve o controle ao programa imediatamente, sem esperar o receptor. Só quando a mensagem é grande (ultrapassa um limiar interno da implementação) o MPI passa a exigir o *rendezvous* — aí sim o `Send` só retorna quando o `Recv` correspondente já foi postado, e um código escrito dessa forma pode travar. As próximas seções tornam esse comportamento **explícito e controlável**, em vez de depender de um detalhe de implementação.

## 2. Os quatro modos de envio bloqueante

O MPI define quatro variantes de envio bloqueante, todas com a **mesma assinatura** de `MPI_Send`, mas semânticas diferentes quanto a *quando* a chamada pode retornar:

| Função | Modo | Quando retorna |
|---|---|---|
| `MPI_Send` | Padrão (*standard*) | Decisão da implementação: pode retornar cedo (buffer interno, protocolo *eager*) ou só depois que o `Recv` for postado (*rendezvous*), dependendo do tamanho da mensagem |
| `MPI_Ssend` | Síncrono (*synchronous*) | Só retorna depois que o processo destino **começou** a receber a mensagem (o `Recv` correspondente já foi postado) |
| `MPI_Bsend` | Bufferizado (*buffered*) | Retorna **imediatamente**, copiando o dado para um buffer de aplicação fornecido pelo programador com `MPI_Buffer_attach` |
| `MPI_Rsend` | Pronto (*ready*) | Retorna imediatamente, mas **exige** que o `Recv` já tenha sido postado antes da chamada — caso contrário, o comportamento é indefinido |

Em todos os quatro casos o `Recv` do lado receptor é sempre o mesmo `MPI_Recv` — o modo de envio é uma escolha exclusiva de quem envia, e é transparente para quem recebe.

### 2.1 `MPI_Ssend` — envio síncrono

`MPI_Ssend` implementa um **rendezvous** explícito: o processo remetente fica bloqueado até que o processo destino efetivamente chame o `MPI_Recv` correspondente (não basta o dado ter sido entregue a um buffer — o receptor precisa ter "aparecido"). É útil quando se quer ter certeza de que emissor e receptor estão sincronizados naquele ponto do programa, ou para depurar códigos que dependem, por engano, do comportamento "por sorte" do `MPI_Send` padrão.

No exemplo abaixo, o processo 1 espera 3 segundos antes de chamar `MPI_Recv`. Repare que o `MPI_Ssend` do processo 0 só retorna depois desses 3 segundos — diferente do que aconteceria com `MPI_Send`.

In [ ]:
%%writefile ssend_demo.c
#include <mpi.h>
#include <stdio.h>
#include <unistd.h>

/* Demonstra o bloqueio do MPI_Ssend: o envio so retorna quando o
   receptor efetivamente chamou o MPI_Recv correspondente (rendezvous). */

int main(void) {
    int rank;
    double t_antes, t_depois;

    MPI_Init(NULL, NULL);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);

    int dado = 42;

    if (rank == 0) {
        printf("[Processo 0] vou chamar MPI_Ssend...\n"); fflush(stdout);
        t_antes = MPI_Wtime();
        MPI_Ssend(&dado, 1, MPI_INT, 1, 0, MPI_COMM_WORLD);
        t_depois = MPI_Wtime();
        printf("[Processo 0] MPI_Ssend retornou apos %.2f s "
               "(so voltou quando o processo 1 chamou o Recv)\n",
               t_depois - t_antes);
    } else if (rank == 1) {
        printf("[Processo 1] vou esperar 3s antes de receber...\n"); fflush(stdout);
        sleep(3);
        int recebido;
        MPI_Recv(&recebido, 1, MPI_INT, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        printf("[Processo 1] recebi %d\n", recebido);
    }

    MPI_Finalize();
    return 0;
}

In [ ]:
!mpicc -o ssend_demo ssend_demo.c
!mpirun --allow-run-as-root --oversubscribe -np 2 ./ssend_demo

### 2.2 `MPI_Bsend` — envio bufferizado

`MPI_Bsend` copia a mensagem para um **buffer de aplicação** — memória alocada pelo próprio programador e registrada junto ao MPI com `MPI_Buffer_attach` — e retorna imediatamente, **independentemente** de o receptor estar pronto ou não. A biblioteca MPI se encarrega de enviar o conteúdo do buffer para o destino em algum momento posterior, de forma transparente ao programador.

```c
int MPI_Buffer_attach(void* buffer, int size);
int MPI_Buffer_detach(void* buffer_ptr, int* size_ptr);
```
- `MPI_Buffer_attach`: registra a região de memória que o MPI poderá usar para os próximos `MPI_Bsend`.
- `MPI_Buffer_detach`: **bloqueia** até que todas as mensagens ainda pendentes no buffer tenham sido efetivamente transmitidas, e então libera o buffer — só depois disso a memória pode ser reaproveitada ou liberada com `free`.
- O tamanho do buffer deve ser, no mínimo, o espaço da mensagem mais a constante `MPI_BSEND_OVERHEAD` (custo fixo de contabilidade do MPI por mensagem em trânsito).

No exemplo, note que `MPI_Bsend` retorna em praticamente 0 segundos, mesmo com o processo 1 demorando 3 segundos para chamar `MPI_Recv` — o oposto do que vimos com `MPI_Ssend`.

In [ ]:
%%writefile bsend_demo.c
#include <mpi.h>
#include <stdio.h>
#include <stdlib.h>
#include <unistd.h>

/* Demonstra o MPI_Bsend: o envio copia o dado para um buffer de
   aplicacao (alocado com MPI_Buffer_attach) e retorna IMEDIATAMENTE,
   mesmo que o receptor ainda nao tenha chamado o Recv. */

int main(void) {
    int rank;
    double t_antes, t_depois;

    MPI_Init(NULL, NULL);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);

    int dado = 7;

    /* Tamanho do buffer = espaco para a mensagem + overhead fixo do MPI
       por mensagem em transito (MPI_BSEND_OVERHEAD). */
    int tam_buffer = sizeof(int) + MPI_BSEND_OVERHEAD;
    void *buffer = malloc(tam_buffer);

    if (rank == 0) {
        MPI_Buffer_attach(buffer, tam_buffer);

        printf("[Processo 0] vou chamar MPI_Bsend...\n"); fflush(stdout);
        t_antes = MPI_Wtime();
        MPI_Bsend(&dado, 1, MPI_INT, 1, 0, MPI_COMM_WORLD);
        t_depois = MPI_Wtime();
        printf("[Processo 0] MPI_Bsend retornou apos %.2f s "
               "(nao esperou o processo 1 estar pronto para receber)\n",
               t_depois - t_antes);

        /* Bloqueia ate que todas as mensagens em buffer sejam enviadas
           de fato pela rede; so entao o buffer pode ser liberado. */
        MPI_Buffer_detach(&buffer, &tam_buffer);
        free(buffer);
    } else if (rank == 1) {
        printf("[Processo 1] vou esperar 3s antes de receber...\n"); fflush(stdout);
        sleep(3);
        int recebido;
        MPI_Recv(&recebido, 1, MPI_INT, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        printf("[Processo 1] recebi %d\n", recebido);
        free(buffer);
    }

    MPI_Finalize();
    return 0;
}

In [ ]:
!mpicc -o bsend_demo bsend_demo.c
!mpirun --allow-run-as-root --oversubscribe -np 2 ./bsend_demo

### 2.3 `MPI_Rsend` — envio pronto

`MPI_Rsend` ("*ready send*") é uma otimização: ele **assume** que o `MPI_Recv` (ou `MPI_Irecv`) correspondente **já foi postado** pelo destino no momento em que é chamado, e por isso pode pular etapas internas de handshake que os outros modos precisam fazer, sendo potencialmente mais rápido. Se essa premissa não for verdadeira — se o receptor ainda não postou o `Recv` — o **comportamento é indefinido** (a mensagem pode se perder, ou o programa pode se comportar de forma imprevisível).

Por isso, `MPI_Rsend` só deve ser usado quando o programador consegue **garantir**, pela lógica do próprio algoritmo, que o receive já está pronto. No exemplo abaixo, essa garantia vem de um `MPI_Irecv` (envio não bloqueante, visto na próxima seção) postado pelo processo 1 **antes** de uma `MPI_Barrier` — como todos os processos só passam da barreira depois que todos chegaram nela, o processo 0 tem certeza de que o `Irecv` já existe quando chama o `Rsend`.

In [ ]:
%%writefile rsend_demo.c
#include <mpi.h>
#include <stdio.h>

/* Demonstra o MPI_Rsend: so e valido chama-lo quando o receptor JA
   postou o MPI_Recv (ou MPI_Irecv) correspondente. Se essa precondicao
   nao for garantida, o comportamento e indefinido. Por isso usamos
   MPI_Irecv + MPI_Barrier para garantir que o receive ja foi postado
   antes do Rsend ser chamado. */

int main(void) {
    int rank;

    MPI_Init(NULL, NULL);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);

    int dado_enviado = 99, dado_recebido = -1;
    MPI_Request req_recv;

    if (rank == 1) {
        /* Posta o receive ANTES da barreira, para garantir que ja esta
           pronto quando o processo 0 chamar o Rsend. */
        MPI_Irecv(&dado_recebido, 1, MPI_INT, 0, 0, MPI_COMM_WORLD, &req_recv);
    }

    /* Sincroniza: garante que o Irecv do processo 1 ja foi postado
       antes de qualquer processo prosseguir. */
    MPI_Barrier(MPI_COMM_WORLD);

    if (rank == 0) {
        printf("[Processo 0] receive do processo 1 ja esta postado; "
               "chamando MPI_Rsend...\n"); fflush(stdout);
        MPI_Rsend(&dado_enviado, 1, MPI_INT, 1, 0, MPI_COMM_WORLD);
        printf("[Processo 0] MPI_Rsend concluido\n");
    } else if (rank == 1) {
        MPI_Wait(&req_recv, MPI_STATUS_IGNORE);
        printf("[Processo 1] recebi %d via Rsend\n", dado_recebido);
    }

    MPI_Finalize();
    return 0;
}

In [ ]:
!mpicc -o rsend_demo rsend_demo.c
!mpirun --allow-run-as-root --oversubscribe -np 2 ./rsend_demo

## 3. Comunicação não bloqueante — `MPI_Isend` / `MPI_Irecv`

Todas as funções vistas até aqui são **bloqueantes**: a chamada só devolve o controle ao programa quando é seguro reaproveitar o buffer (no envio) ou quando o dado já chegou (no recebimento). Isso significa que o processo fica **parado**, sem poder fazer nenhum outro trabalho, enquanto a comunicação está em andamento.

As versões **não bloqueantes** — `MPI_Isend` ("*immediate send*") e `MPI_Irecv` — resolvem isso: a chamada apenas **registra a intenção** de enviar/receber e retorna imediatamente, devolvendo um identificador (`MPI_Request`) que representa a operação em andamento. O programa pode então fazer outro trabalho útil enquanto a comunicação acontece em paralelo, "por trás dos panos", e só precisa esperar de fato quando o resultado for necessário.

```c
int MPI_Isend(const void* buf, int count, MPI_Datatype datatype,
              int dest, int tag, MPI_Comm comm, MPI_Request* request);

int MPI_Irecv(void* buf, int count, MPI_Datatype datatype,
              int source, int tag, MPI_Comm comm, MPI_Request* request);

int MPI_Wait(MPI_Request* request, MPI_Status* status);
int MPI_Test(MPI_Request* request, int* flag, MPI_Status* status);
```
- `request`: identificador da operação assíncrona, preenchido por `Isend`/`Irecv` e consumido por `Wait`/`Test`.
- `MPI_Wait`: **bloqueia** até que a operação associada ao `request` termine.
- `MPI_Test`: **não bloqueia** — apenas verifica se a operação já terminou (`flag = 1`) ou não (`flag = 0`), podendo ser chamado repetidamente em um laço.
- É **proibido** acessar ou modificar o buffer (`buf`) antes de a operação ser confirmada como concluída (via `Wait` ou `Test` com `flag = 1`).

> Existem também `MPI_Issend`, `MPI_Ibsend` e `MPI_Irsend` — as versões não bloqueantes dos outros três modos de envio da Seção 2. A lógica é a mesma: a chamada retorna na hora com um `MPI_Request`, e as garantias semânticas de cada modo (síncrono, bufferizado, pronto) só são checadas quando o `Wait`/`Test` correspondente confirma a conclusão.

O exemplo a seguir reescreve o anel de saudações da Seção 1 com `MPI_Isend`/`MPI_Irecv`: cada processo dispara o envio e o recebimento, faz um cálculo qualquer enquanto a mensagem "viaja", e só então espera (`MPI_Wait`) o resultado — sem correr o risco de *deadlock* discutido no quadro de atenção da Seção 1, já que nenhuma das duas chamadas bloqueia o processo.

In [ ]:
%%writefile isend_irecv_demo.c
#include <mpi.h>
#include <stdio.h>
#include <string.h>

const int MAX_STRING = 100;

/* Mesmo anel de saudacoes do exemplo inicial, mas usando MPI_Isend e
   MPI_Irecv: as chamadas retornam imediatamente, permitindo que cada
   processo faca outro trabalho ("computo util") enquanto a comunicacao
   acontece em segundo plano. So paramos para esperar (MPI_Wait) quando
   realmente precisamos do resultado. */

int main(void) {
    char greeting[MAX_STRING];
    char recebido[MAX_STRING];
    int size, rank;
    MPI_Request req_send, req_recv;

    MPI_Init(NULL, NULL);
    MPI_Comm_size(MPI_COMM_WORLD, &size);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);

    sprintf(greeting, "Greetings from process %d of %d!", rank, size);

    int destino = (rank + 1) % size;
    int origem  = (rank - 1 + size) % size;

    /* Ambas as chamadas retornam na hora: apenas "registram" a
       intencao de enviar/receber e devolvem um MPI_Request. */
    MPI_Isend(greeting, strlen(greeting) + 1, MPI_CHAR, destino, 0, MPI_COMM_WORLD, &req_send);
    MPI_Irecv(recebido, MAX_STRING, MPI_CHAR, origem, 0, MPI_COMM_WORLD, &req_recv);

    /* "Computo util" que pode ser feito enquanto a mensagem viaja */
    long soma = 0;
    for (long i = 0; i < 100000000L; i++) soma += i % 7;
    printf("[Processo %d] fiz um trabalho qualquer enquanto a mensagem "
           "estava a caminho (soma=%ld)\n", rank, soma);

    /* So agora bloqueamos, esperando as operacoes assincronas terminarem */
    MPI_Wait(&req_send, MPI_STATUS_IGNORE);
    MPI_Wait(&req_recv, MPI_STATUS_IGNORE);

    printf("[Processo %d] recebeu: %s\n", rank, recebido);

    MPI_Finalize();
    return 0;
}

In [ ]:
!mpicc -O2 -o isend_irecv_demo isend_irecv_demo.c
!mpirun --allow-run-as-root --oversubscribe -np 4 ./isend_irecv_demo

## 4. `MPI_ANY_SOURCE` e `MPI_ANY_TAG`

Em todos os exemplos anteriores, cada `MPI_Recv` sabia exatamente **de quem** e **com que tag** esperar a mensagem. Mas em padrões do tipo *mestre-trabalhadores* (*master-worker*), o processo mestre muitas vezes não sabe, de antemão, qual trabalhador vai terminar primeiro — e processar as mensagens **na ordem em que elas realmente chegam** (em vez de esperar o processo 1, depois o 2, depois o 3...) evita tempo ocioso.

Para isso, o MPI define dois curingas que podem ser usados **apenas nos parâmetros `source` e `tag` de uma chamada de recebimento** (nunca em um envio):

- `MPI_ANY_SOURCE`: casa com uma mensagem vinda de **qualquer** processo.
- `MPI_ANY_TAG`: casa com uma mensagem de **qualquer** tag.

Como o `Recv` não sabe mais, a priori, quem enviou nem com que tag, essa informação só pode ser descoberta **depois** da chamada, consultando a estrutura `MPI_Status`:

```c
typedef struct {
    int MPI_SOURCE;  /* rank de quem realmente enviou a mensagem   */
    int MPI_TAG;      /* tag que a mensagem realmente carregava     */
    int MPI_ERROR;    /* codigo de erro, se aplicavel                */
    /* ... campos internos usados por MPI_Get_count ... */
} MPI_Status;

int MPI_Get_count(const MPI_Status* status, MPI_Datatype datatype, int* count);
```
`MPI_Get_count` é útil quando o número de elementos efetivamente recebido pode ser **menor** do que o tamanho do buffer fornecido no `Recv` (que é apenas um limite superior).

No exemplo, cada trabalhador espera um tempo aleatório e diferente antes de enviar seu rank ao processo 0, usando uma tag também baseada em seu rank. O processo 0 não sabe qual vai chegar primeiro nem qual tag cada um vai usar — por isso recebe com `MPI_ANY_SOURCE` e `MPI_ANY_TAG`, e só depois, pelo `status`, descobre quem enviou o quê.

In [ ]:
%%writefile any_source_tag_demo.c
#include <mpi.h>
#include <stdio.h>
#include <stdlib.h>
#include <unistd.h>

/* Cada processo "trabalhador" (rank != 0) espera um tempo aleatorio e
   entao envia seu rank para o processo 0, usando como tag o dobro do
   seu proprio rank (so para ilustrar que a tag carrega informacao).
   O processo 0 NAO sabe de antemao qual trabalhador vai terminar
   primeiro nem qual tag cada um vai usar -- por isso ele recebe com
   MPI_ANY_SOURCE e MPI_ANY_TAG, e descobre quem enviou (e com que tag)
   consultando o MPI_Status depois do MPI_Recv. */

int main(void) {
    int rank, size;

    MPI_Init(NULL, NULL);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    if (rank == 0) {
        /* Recebe uma mensagem de cada um dos (size - 1) trabalhadores,
           na ordem em que elas realmente chegarem. */
        for (int i = 0; i < size - 1; i++) {
            int valor;
            MPI_Status status;

            MPI_Recv(&valor, 1, MPI_INT,
                      MPI_ANY_SOURCE, MPI_ANY_TAG,
                      MPI_COMM_WORLD, &status);

            /* O status revela quem enviou de fato e com qual tag,
               informacoes que nao foram especificadas no Recv. */
            int origem = status.MPI_SOURCE;
            int tag    = status.MPI_TAG;

            int contagem;
            MPI_Get_count(&status, MPI_INT, &contagem);

            printf("[Processo 0] recebi valor=%d do processo %d "
                   "(tag=%d, %d elemento(s))\n",
                   valor, origem, tag, contagem);
            fflush(stdout);
        }
    } else {
        /* Cada trabalhador "demora" um tempo diferente e imprevisivel */
        unsigned int semente = (unsigned int) (rank * 1234567);
        int atraso = rand_r(&semente) % 3;   /* 0, 1 ou 2 segundos */
        sleep(atraso);

        int tag_usada = rank * 2;
        MPI_Send(&rank, 1, MPI_INT, 0, tag_usada, MPI_COMM_WORLD);
    }

    MPI_Finalize();
    return 0;
}

In [ ]:
!mpicc -o any_source_tag_demo any_source_tag_demo.c
!mpirun --allow-run-as-root --oversubscribe -np 5 ./any_source_tag_demo

> **Repare** que a ordem em que os valores chegam (3, 4, 1, 2 nesta execução) depende dos atrasos aleatórios de cada trabalhador — rodando a célula novamente, a ordem provavelmente muda. Essa é exatamente a vantagem do `MPI_ANY_SOURCE`: o mestre processa cada mensagem assim que ela chega, sem impor uma ordem artificial.

## 5. Medição de tempo com `MPI_Wtime`

O MPI oferece um cronômetro próprio, portátil entre implementações e plataformas:

```c
double MPI_Wtime(void);
double MPI_Wtick(void);
```
- `MPI_Wtime()` retorna o número de segundos (como `double`) decorridos desde algum instante de referência no passado — esse instante **não é especificado** pelo padrão e pode variar entre processos, então o valor de uma única chamada não tem significado isolado; o que importa é sempre a **diferença** entre duas chamadas (`fim - inicio`).
- `MPI_Wtick()` retorna a **resolução** do relógio: o menor intervalo de tempo que `MPI_Wtime` consegue distinguir, em segundos.

**Por que sincronizar com `MPI_Barrier` antes de medir?** Se cada processo simplesmente chamasse `MPI_Wtime()` assim que chegasse naquele ponto do código, processos que chegaram atrasados começariam a medir a partir de um instante diferente dos que chegaram cedo — a medição de tempo ficaria contaminada por esse desalinhamento inicial, e não refletiria o tempo do algoritmo paralelo em si. Colocar uma `MPI_Barrier` imediatamente antes de `t_inicio = MPI_Wtime()` garante que **todos os processos começam a cronometrar a partir do mesmo ponto lógico** — o instante em que o processo mais lento também chegou ali.

Depois de medir o tempo **local** de cada processo, é comum combinar os resultados com `MPI_Reduce` usando `MPI_MAX`: o programa paralelo como um todo só termina quando o processo **mais lento** termina, então o tempo máximo entre os processos é o número mais representativo do tempo total do algoritmo (medir só o tempo do processo 0, por exemplo, poderia subestimar o tempo real caso outro processo seja o gargalo).

> **Cuidado:** por padrão, o MPI não garante que os relógios de processos diferentes estejam sincronizados entre si (a menos que a implementação exponha o atributo `MPI_WTIME_IS_GLOBAL` como verdadeiro, o que é incomum fora de alguns supercomputadores). Isso não afeta o padrão usado aqui — cada processo mede seu próprio intervalo local (`fim - início`), nunca comparando o relógio de um processo com o de outro diretamente.

O exemplo abaixo cronometra 1000 repetições do anel `Send`/`Recv` da Seção 1.

In [ ]:
%%writefile timing_demo.c
#include <mpi.h>
#include <stdio.h>
#include <string.h>

const int MAX_STRING = 100;
const int N_REPETICOES = 1000;

/* Mede o tempo do anel de comunicacao (Send/Recv) do exemplo inicial,
   repetido N_REPETICOES vezes, usando MPI_Wtime. */

int main(void) {
    char greeting[MAX_STRING];
    int size, rank;

    MPI_Init(NULL, NULL);
    MPI_Comm_size(MPI_COMM_WORLD, &size);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);

    if (rank == 0) {
        /* Resolucao do relogio: menor intervalo de tempo distinguivel
           por MPI_Wtime nesta maquina. */
        printf("[Processo 0] resolucao do MPI_Wtime (MPI_Wtick): %e s\n",
               MPI_Wtick());
    }

    sprintf(greeting, "Greetings from process %d of %d!", rank, size);

    int destino = (rank + 1) % size;
    int origem  = (rank - 1 + size) % size;

    /* IMPORTANTE: sincroniza todos os processos com uma barreira antes
       de iniciar a medicao. Sem isso, processos que chegam atrasados
       "roubariam" tempo da medicao de quem chegou cedo, e cada processo
       estaria medindo a partir de um instante de partida diferente. */
    MPI_Barrier(MPI_COMM_WORLD);
    double t_inicio = MPI_Wtime();

    char recebido[MAX_STRING];
    for (int r = 0; r < N_REPETICOES; r++) {
        MPI_Send(greeting, strlen(greeting) + 1, MPI_CHAR, destino, 0, MPI_COMM_WORLD);
        MPI_Recv(recebido, MAX_STRING, MPI_CHAR, origem, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
    }

    double t_fim_local = MPI_Wtime();
    double tempo_local = t_fim_local - t_inicio;

    /* Cada processo mede seu proprio tempo; para reportar um unico
       numero representativo do algoritmo como um todo, usamos o PIOR
       caso (MPI_MAX) entre todos os processos -- o tempo total so
       termina quando o processo mais lento tambem terminou. */
    double tempo_max;
    MPI_Reduce(&tempo_local, &tempo_max, 1, MPI_DOUBLE, MPI_MAX, 0, MPI_COMM_WORLD);

    printf("[Processo %d] tempo local para %d repeticoes: %.6f s\n",
           rank, N_REPETICOES, tempo_local);

    if (rank == 0) {
        printf("\n[Processo 0] tempo maximo entre os %d processos: %.6f s "
               "(%.3f microssegundos por par send/recv)\n",
               size, tempo_max, 1e6 * tempo_max / N_REPETICOES);
    }

    MPI_Finalize();
    return 0;
}

In [ ]:
!mpicc -O2 -o timing_demo timing_demo.c
!mpirun --allow-run-as-root --oversubscribe -np 4 ./timing_demo

## 6. Resumo das funções apresentadas

| Função | Categoria | Efeito |
|---|---|---|
| `MPI_Send` | Envio bloqueante — padrão | Pode retornar antes ou depois do `Recv`, a critério da implementação |
| `MPI_Ssend` | Envio bloqueante — síncrono | Só retorna depois que o `Recv` correspondente foi postado |
| `MPI_Bsend` | Envio bloqueante — bufferizado | Retorna imediatamente; copia o dado para um buffer de aplicação |
| `MPI_Rsend` | Envio bloqueante — pronto | Retorna imediatamente; exige que o `Recv` já esteja postado |
| `MPI_Recv` | Recebimento bloqueante | Só retorna quando a mensagem completa chegou |
| `MPI_Isend` / `MPI_Irecv` | Não bloqueante | Retornam na hora com um `MPI_Request`; use `MPI_Wait`/`MPI_Test` para confirmar |
| `MPI_ANY_SOURCE` | Curinga (só em `Recv`) | Casa com mensagem de qualquer processo |
| `MPI_ANY_TAG` | Curinga (só em `Recv`) | Casa com mensagem de qualquer tag |
| `MPI_Get_count` | Consulta | Número real de elementos recebidos, a partir do `MPI_Status` |
| `MPI_Wtime` / `MPI_Wtick` | Medição de tempo | Cronômetro portátil do MPI e sua resolução |
| `MPI_Barrier` | Sincronização | Necessária antes de medir tempo, para alinhar o instante de partida |

## 7. Exercícios

Implemente os exercícios abaixo em **C**, usando o mesmo padrão `%%writefile` + `mpicc` + `mpirun` utilizado nos exemplos acima. Cada exercício tem duas células logo após o enunciado: a primeira já contém o `%%writefile` com o nome do arquivo — escreva seu código C nela — e a segunda já contém os comandos de compilação e execução, prontos para rodar.

### Exercício 1 — Send padrão com mensagens grandes

Modifique o exemplo da Seção 1 (`greetings.c`) para enviar um vetor de `double` bem maior (por exemplo, 1 milhão de elementos) em vez de uma string curta, mantendo a mesma estrutura de anel (`Send` seguido de `Recv`, todo processo enviando antes de receber). Observe o que acontece — o programa ainda termina normalmente, ou trava? Relacione o resultado com o quadro de atenção da Seção 1 sobre os protocolos *eager* e *rendezvous*, e proponha uma correção usando `MPI_Isend`/`MPI_Irecv` (Seção 3) ou alternando a ordem de `Send`/`Recv` entre processos de rank par e ímpar.

In [ ]:
%%writefile exercicio1.c


In [ ]:
!mpicc -o exercicio1 exercicio1.c
!mpirun --allow-run-as-root --oversubscribe -np 4 ./exercicio1

### Exercício 2 — Comparando `MPI_Ssend` e `MPI_Isend` + `MPI_Wait`

Usando dois processos, escreva duas versões de um mesmo envio de um vetor grande (por exemplo, 10 milhões de inteiros): uma com `MPI_Ssend`, outra com `MPI_Isend` seguido imediatamente de `MPI_Wait`. Cronometre as duas com `MPI_Wtime` (lembre-se do `MPI_Barrier` antes de medir) e repita o envio algumas dezenas de vezes para obter uma média. As duas formas devem ter tempo total parecido — explique por quê, e o que mudaria se, entre o `Isend` e o `Wait`, o processo fizesse um cálculo demorado antes de esperar.

In [ ]:
%%writefile exercicio2.c


In [ ]:
!mpicc -o exercicio2 exercicio2.c
!mpirun --allow-run-as-root --oversubscribe -np 2 ./exercicio2

### Exercício 3 — Mestre-trabalhadores com `MPI_ANY_SOURCE`

Cada processo trabalhador (rank ≠ 0) deve calcular a soma dos números de `1` até `(rank * 100000)` e enviar o resultado ao processo 0, usando como tag o próprio rank. O processo 0 deve receber os `size - 1` resultados com `MPI_ANY_SOURCE` e `MPI_ANY_TAG`, imprimindo, para cada um, o rank de origem (obtido via `status.MPI_SOURCE`, já que a tag é igual ao rank) e o valor recebido, na ordem em que realmente chegarem. Ao final, o processo 0 deve imprimir a soma de todos os resultados recebidos.

In [ ]:
%%writefile exercicio3.c


In [ ]:
!mpicc -o exercicio3 exercicio3.c
!mpirun --allow-run-as-root --oversubscribe -np 5 ./exercicio3

### Exercício 4 — Cronometrando uma redução com `MPI_Barrier`

Escreva um programa em que cada processo gera um valor (por exemplo, `rank * rank`) e todos os processos calculam a soma total com `MPI_Reduce` (`MPI_SUM`, raiz 0), repetindo essa redução 10000 vezes dentro de um laço. Meça o tempo total com `MPI_Wtime`, comparando o resultado de duas versões do programa: uma que coloca `MPI_Barrier` antes de `t_inicio = MPI_Wtime()`, e outra que **não** coloca. Rode cada versão algumas vezes e discuta se a diferença entre elas fica mais evidente com poucos processos ou com muitos.

In [ ]:
%%writefile exercicio4.c


In [ ]:
!mpicc -o exercicio4 exercicio4.c
!mpirun --allow-run-as-root --oversubscribe -np 4 ./exercicio4